# 🧠 CNN: Steam žaidimo žanras iš viršelio paveikslėlio
**Užduotis 1.4** - konvoliucinio tinklo (CNN) Colab'o perdarymas su savo duomenimis.

## Idėja
Mūsų projektas atsako į klausimą „koks tai žaidimas?“. Tekstą analizuoja Gemini, o čia tą patį klausimą sprendžiame **iš vaizdo**: ar CNN pagal Steam parduotuvės viršelį (`header.jpg`) atpažins žanrą (Racing / Strategy / Sports / RPG)?

## Kaip veikia CNN (tai, ką reikia paaiškinti gynimo metu)
| Sluoksnis | Ką daro |
|---|---|
| `Rescaling(1/255)` | pikselių reikšmes iš 0-255 sumažina į 0-1, kad mokymas būtų stabilesnis |
| `RandomFlip / RandomZoom / RandomContrast` | **duomenų augmentacija**: mokymo metu kiekvieną kartą šiek tiek pakeičia paveikslėlį, taip tinklas mažiau įsimena konkrečius vaizdus |
| `Conv2D(filtrai, 3x3)` | slenka mažu langu (filtru) per vaizdą ir ieško raštų: kraštų, spalvų, vėliau formų (automobilis, kardas, kamuolys) |
| `MaxPooling2D` | iš kiekvieno 2x2 langelio palieka didžiausią reikšmę: mažina dydį, daro atpažinimą atsparesnį poslinkiams |
| `GlobalAveragePooling2D` | kiekvieną požymių žemėlapį suvidurkina iki vieno skaičiaus (mažiau parametrų nei `Flatten`) |
| `Dense + Dropout` | klasifikatorius; `Dropout` atsitiktinai išjungia dalį neuronų, kovoja su permokymu |
| `Dense(softmax)` | tikimybė kiekvienam žanrui, suma = 1 |

**Prieš paleidžiant:** `Runtime → Change runtime type → T4 GPU`.

## 1. Duomenų rinkinio sukūrimas
Duomenis renkame patys iš viešų šaltinių: **SteamSpy API** pateikia žaidimus pagal žanrą, o paveikslėlius siunčiame iš Steam CDN. Pasirenkame populiariausius žaidimus (jų viršeliai tvarkingi) ir išmetame tuos, kurie priklauso keliems pasirinktiems žanrams. Taip klasės neprieštarauja viena kitai.

In [ ]:
import os, time, random, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
SEED = 42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED); sns.set_theme(style="whitegrid")

GENRES = ["Racing", "Strategy", "Sports", "RPG"]   # <-- galima keisti (SteamSpy žanrai: Action, Adventure, Casual, Indie, RPG, Simulation, Strategy, Racing, Sports ...)
PER_GENRE = 250
ROOT = "/content/steam_genres"
HEADER = "https://cdn.cloudflare.steamstatic.com/steam/apps/{}/header.jpg"

def top_apps(genre, k):
    j = requests.get("https://steamspy.com/api.php", params={"request": "genre", "genre": genre}, timeout=60).json()
    apps = sorted(j.values(), key=lambda a: a["positive"] + a["negative"], reverse=True)   # populiariausi turi tvarkingus viršelius
    return [(a["appid"], a["name"]) for a in apps[:k * 2]]

def download(args):
    appid, genre = args
    out = f"{ROOT}/{genre}/{appid}.jpg"
    if os.path.exists(out): return out
    try:
        r = requests.get(HEADER.format(appid), timeout=15)
        if r.status_code == 200 and len(r.content) > 5000:
            open(out, "wb").write(r.content); return out
    except Exception:
        pass
    return None

if not os.path.isdir(ROOT):
    cand = {}
    for g in GENRES:
        cand[g] = top_apps(g, PER_GENRE); time.sleep(2)
    # pašaliname žaidimus, priklausančius keliems pasirinktiems žanrams (kad klasės nesidubliuotų)
    cnt = {}
    for g, lst in cand.items():
        for appid, _ in lst: cnt[appid] = cnt.get(appid, 0) + 1
    jobs = []
    for g, lst in cand.items():
        os.makedirs(f"{ROOT}/{g}", exist_ok=True)
        jobs += [(a, g) for a, _ in [x for x in lst if cnt[x[0]] == 1]]
    with ThreadPoolExecutor(16) as ex: res = list(ex.map(download, jobs))
    # išlyginame klases: palieka tiek vaizdų, kiek yra mažiausioje klasėje (iki PER_GENRE)
    for g in GENRES:
        files = sorted(os.listdir(f"{ROOT}/{g}"))
        for f in files[PER_GENRE:]: os.remove(f"{ROOT}/{g}/{f}")

paths, labels = [], []
for i, g in enumerate(GENRES):
    fs = sorted(os.listdir(f"{ROOT}/{g}")); paths += [f"{ROOT}/{g}/{f}" for f in fs]; labels += [i] * len(fs)
paths, labels = np.array(paths), np.array(labels)
print(pd.Series(labels).map(dict(enumerate(GENRES))).value_counts())

## 2. Skaidymas ir `tf.data` vamzdynas
70 % mokymui, 15 % validacijai (hiperparametrams ir ankstyvam stabdymui), 15 % galutiniam testui, kurio mokymo metu **nematome**. `stratify` išlaiko vienodą klasių proporciją visose dalyse.

In [ ]:
IMG_SIZE = (128, 256)   # viršeliai ~460x215, išlaikome santykį maždaug 1:2
BATCH = 32; AUTOTUNE = tf.data.AUTOTUNE

p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print("train/val/test:", len(p_tr), len(p_va), len(p_te))

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(p, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y)).map(load, num_parallel_calls=AUTOTUNE).cache()
    if training: ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(p_tr, y_tr, True), make_ds(p_va, y_va), make_ds(p_te, y_te)

imgs, labs = next(iter(train_ds))
plt.figure(figsize=(12, 5))
for i in range(8):
    ax = plt.subplot(2, 4, i + 1); ax.imshow(imgs[i].numpy().astype("uint8")); ax.set_title(GENRES[int(labs[i])]); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Modelis
Trys konvoliuciniai blokai (32 → 64 → 128 filtrų): ankstyvi sluoksniai mato paprastus raštus, gilesni sudėtingesnius.

In [ ]:
aug = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomZoom(0.1),
                           tf.keras.layers.RandomContrast(0.1)], name="augmentation")
def conv_block(f):
    return [tf.keras.layers.Conv2D(f, 3, padding="same", activation="relu"),
            tf.keras.layers.BatchNormalization(), tf.keras.layers.MaxPooling2D()]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SIZE + (3,)),
    aug, tf.keras.layers.Rescaling(1 / 255),
    *conv_block(32), *conv_block(64), *conv_block(128),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(GENRES), activation="softmax")])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Mokymas
`EarlyStopping` sustabdo, kai validacijos klaida nustoja mažėti, o `ReduceLROnPlateau` sumažina mokymosi greitį, kai pasiekiama „plynaukštė“.

In [ ]:
cbs = [tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)]
hist = model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=cbs)

In [ ]:
h = pd.DataFrame(hist.history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
h[["loss", "val_loss"]].plot(ax=ax[0], title="Klaida"); h[["accuracy", "val_accuracy"]].plot(ax=ax[1], title="Tikslumas")
plt.show()

## 5. Vertinimas testo aibėje
Painiavos matrica parodo, kuriuos žanrus tinklas painioja. Tikėtina, kad Racing ir Sports atskiria geriau (aiškūs vaizdiniai motyvai), o Strategy ir RPG painioja (panaši fantazijos estetika).

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Testo tikslumas: {test_acc:.3f} (atsitiktinis spėjimas: {1/len(GENRES):.2f})")
y_pred = np.concatenate([model.predict(x, verbose=0).argmax(1) for x, _ in test_ds])
y_true = np.concatenate([y.numpy() for _, y in test_ds])
print(classification_report(y_true, y_pred, target_names=GENRES))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=GENRES, yticklabels=GENRES)
plt.xlabel("Prognozė"); plt.ylabel("Tikra"); plt.show()
CNN_TEST_ACC = test_acc

## 6. Ką tinklas „mato“: filtrų aktyvacijos
Parodome pirmojo konvoliucinio sluoksnio išėjimą vienam paveikslėliui, o šviesios sritys rodo, kur filtras „aptiko“ savo raštą.

In [ ]:
x0 = imgs[:1]
first_conv = [l for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)][0]
feat = tf.keras.Model(model.inputs, first_conv.output)(x0).numpy()[0]   # modelis pats atlieka Rescaling
plt.figure(figsize=(12, 4)); plt.subplot(1, 5, 1); plt.imshow(x0[0].numpy().astype("uint8")); plt.axis("off"); plt.title("originalas")
for i in range(4):
    plt.subplot(1, 5, i + 2); plt.imshow(feat[..., i], cmap="viridis"); plt.axis("off"); plt.title(f"filtras {i}")
plt.show()

## 7. Išvados
* CNN iš nulio su ~700 paveikslėlių pasiekia gerokai geriau nei atsitiktinį spėjimą, bet duomenų mažai, todėl tikslumas ribotas (galite pamatyti didelį atotrūkį tarp `accuracy` ir `val_accuracy`: permokymas).
* Kitas žingsnis `05_transfer_learning.ipynb`: tą pačią užduotį sprendžiame su iš anksto apmokytu tinklu.